# Gaussian rainfall diagnostics

This notebook examines the Gaussian assumption for monthly station rainfall. It retains the monthly histogram diagnostic for one representative station and adds station-by-month heatmaps of unbiased sample skewness and excess kurtosis. For a Gaussian distribution, both quantities are zero.

In [ ]:
%matplotlib inline

from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap, TwoSlopeNorm

mpl.rcParams['figure.dpi'] = 150
mpl.rcParams['font.family'] = 'Myriad Pro'

N_MONTHS = 12
MONTH_NAMES = np.array(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                        'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

In [ ]:
rain_obs = np.loadtxt('data/sta_monthly.csv')
station_lookup = pd.read_csv('data/sta_lookup_new.csv', header=None)
station_names = station_lookup.iloc[:, 0].to_numpy()

N_STATIONS = rain_obs.shape[1]
N_YEARS = rain_obs.shape[0] // N_MONTHS
STATION_LABELS = np.array([f'Station {idx + 1}' for idx in range(N_STATIONS)])

assert rain_obs.shape == (N_YEARS * N_MONTHS, N_STATIONS)
assert station_names.size == N_STATIONS

station_key = pd.DataFrame({
    'station': STATION_LABELS,
    'station_name': station_names,
})
print(f'Loaded {N_YEARS} years, {N_MONTHS} months, and {N_STATIONS} stations.')
station_key

## Monthly histograms for a representative station

The first station is retained as the representative histogram example from the original notebook. Each panel overlays the fitted Gaussian density and reports the Shapiro-Wilk p-value.

In [ ]:
station_idx = 0
station_monthly = np.column_stack([
    rain_obs[month_idx::N_MONTHS, station_idx]
    for month_idx in range(N_MONTHS)
])

ipcc_blue = '#70A0CD'
ipcc_orange = '#C47900'

fig, axes = plt.subplots(4, 3, figsize=(12, 12))
axes = axes.ravel()

for month_idx, ax in enumerate(axes):
    month_data = station_monthly[:, month_idx]
    sample_mean = np.mean(month_data)
    sample_std = np.std(month_data, ddof=1)
    _, shapiro_p = stats.shapiro(month_data)

    ax.hist(
        month_data, bins=8, density=True, alpha=0.7,
        color=ipcc_blue, edgecolor='black', linewidth=0.5
    )
    x_values = np.linspace(month_data.min(), month_data.max(), 200)
    ax.plot(
        x_values, stats.norm.pdf(x_values, sample_mean, sample_std),
        color=ipcc_orange, linewidth=2
    )
    ax.set_title(MONTH_NAMES[month_idx], fontweight='bold')
    ax.set_xlabel('Rainfall [mm]')
    ax.set_ylabel('Density')
    ax.text(
        0.97, 0.95,
        f'$\mu$={sample_mean:.1f} mm\n$\sigma$={sample_std:.1f} mm\nShapiro-Wilk $p$={shapiro_p:.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=8,
        bbox={'facecolor': 'white', 'alpha': 0.75, 'edgecolor': 'none'}
    )
    ax.grid(True, alpha=0.25)

fig.suptitle(f'Monthly rainfall distributions: {station_names[station_idx]}', y=1.01)
fig.tight_layout()
fig.savefig('rain_hist.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

## Skewness and excess-kurtosis diagnostics for every station-month

Skewness and kurtosis are calculated from the 40 annual observations available for each station and calendar month. `bias=False` applies the finite-sample corrections, while `fisher=True` reports excess kurtosis so the Gaussian reference value is zero.

In [ ]:
skewness = np.zeros((N_STATIONS, N_MONTHS))
excess_kurtosis = np.zeros((N_STATIONS, N_MONTHS))
shapiro_p_values = np.zeros((N_STATIONS, N_MONTHS))
SHAPIRO_ALPHA = 0.05

for station_idx in range(N_STATIONS):
    for month_idx in range(N_MONTHS):
        station_month_data = rain_obs[month_idx::N_MONTHS, station_idx]
        skewness[station_idx, month_idx] = stats.skew(
            station_month_data, bias=False
        )
        excess_kurtosis[station_idx, month_idx] = stats.kurtosis(
            station_month_data, fisher=True, bias=False
        )
        shapiro_p_values[station_idx, month_idx] = stats.shapiro(
            station_month_data
        ).pvalue

skewness_table = pd.DataFrame(
    skewness, index=STATION_LABELS, columns=MONTH_NAMES
)
excess_kurtosis_table = pd.DataFrame(
    excess_kurtosis, index=STATION_LABELS, columns=MONTH_NAMES
)
shapiro_fail_to_reject = shapiro_p_values >= SHAPIRO_ALPHA
shapiro_p_value_table = pd.DataFrame(
    shapiro_p_values, index=STATION_LABELS, columns=MONTH_NAMES
)

print('Skewness range:', np.nanmin(skewness), 'to', np.nanmax(skewness))
print(
    'Excess-kurtosis range:',
    np.nanmin(excess_kurtosis), 'to', np.nanmax(excess_kurtosis)
)

In [ ]:
skewness_table.round(3)

In [ ]:
excess_kurtosis_table.round(3)

## Binary Shapiro-Wilk decision heatmap

At $\alpha=0.05$, red cells reject the Gaussian null hypothesis and green cells fail to reject it. The colorbar is discrete rather than continuous.

In [ ]:
decision_cmap = ListedColormap(['#B40426', '#006837'])
decision_bounds = np.array([-0.5, 0.5, 1.5])
decision_norm = BoundaryNorm(decision_bounds, decision_cmap.N)

fig, ax = plt.subplots(figsize=(10.5, 7.5))
decision_image = ax.imshow(
    shapiro_fail_to_reject.astype(int),
    aspect='auto', interpolation='nearest',
    cmap=decision_cmap, norm=decision_norm
)

ax.set_title(
    f'Shapiro-Wilk normality decisions (α = {SHAPIRO_ALPHA:.2f})',
    fontweight='bold'
)
ax.set_xticks(np.arange(N_MONTHS))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(np.arange(N_STATIONS))
ax.set_yticklabels(STATION_LABELS)
ax.set_xlabel('Calendar month')
ax.set_ylabel('Station')

ax.set_xticks(np.arange(-0.5, N_MONTHS, 1), minor=True)
ax.set_yticks(np.arange(-0.5, N_STATIONS, 1), minor=True)
ax.grid(which='minor', color='white', linewidth=0.7, alpha=0.65)
ax.tick_params(which='minor', bottom=False, left=False)

colorbar = fig.colorbar(
    decision_image, ax=ax, ticks=[0, 1],
    boundaries=decision_bounds, spacing='uniform',
    fraction=0.046, pad=0.04
)
colorbar.ax.set_yticklabels([
    'Reject (not Gaussian)',
    'Fail to reject',
])
colorbar.set_label('Shapiro-Wilk decision')

fig.tight_layout()
fig.savefig('shapiro_wilk_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
def plot_station_month_heatmap(ax, values, title, colorbar_label):
    color_limit = np.nanmax(np.abs(values))
    if not np.isfinite(color_limit) or color_limit == 0:
        color_limit = 1.0

    image = ax.imshow(
        values, aspect='auto', interpolation='nearest', cmap='RdBu_r',
        norm=TwoSlopeNorm(vmin=-color_limit, vcenter=0, vmax=color_limit)
    )
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(np.arange(N_MONTHS))
    ax.set_xticklabels(MONTH_NAMES, rotation=45, ha='right')
    ax.set_yticks(np.arange(N_STATIONS))
    ax.set_yticklabels(STATION_LABELS)
    ax.set_xlabel('Calendar month')

    ax.set_xticks(np.arange(-0.5, N_MONTHS, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, N_STATIONS, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=0.6, alpha=0.7)
    ax.tick_params(which='minor', bottom=False, left=False)

    colorbar = ax.figure.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    colorbar.set_label(colorbar_label)
    return image


fig, axes = plt.subplots(1, 2, figsize=(14, 7.5), sharey=True)
plot_station_month_heatmap(
    axes[0], skewness, '(a) Skewness', 'Unbiased sample skewness'
)
plot_station_month_heatmap(
    axes[1], excess_kurtosis, '(b) Excess kurtosis',
    'Unbiased excess kurtosis'
)
axes[0].set_ylabel('Station')
axes[1].tick_params(axis='y', labelleft=True)

fig.suptitle('Station-month rainfall distribution shape', fontweight='bold')
fig.tight_layout()
fig.savefig(
    FIGURE_DIR / 'skewness_kurtosis_heatmaps.png',
    dpi=300, bbox_inches='tight'
)
plt.show()
plt.close(fig)